# SolarVoix — Generation of synthetic data

**Notebook Fabric :** `NB_Data_Generation`

**Lakehouse requis :** attachez `LH_SolarVoix`as the default Lakehouse before execution.

Run all three cells in order. The last cell creates`Files/audio_calls/`and`Files/metadata/calls_metadata.csv`.


## Cell 1 — Installation of dependencies


In [ ]:
# === CELLULE 1 : Installation des dépendances ===
#
# OBJECTIF : Installer les bibliothèques nécessaires pour générer les données
#            synthétiques et effectuer l'analyse IA sans clé externe.
#
# %pip install s'exécute dans l'environnement du cluster Spark.
# L'installation est temporaire (durée de la session Spark).
# Pour persister, utilisez Environment items dans Fabric.
#
# faker    → génère des noms, villes, dates aléatoires réalistes
# pydub    → manipulation audio (création de fichiers WAV/MP3 vides)
# scikit-learn et joblib seront installés dans le notebook d'analyse


import subprocess, sys
pkgs = ["faker", "pydub"]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs, check=True)
print("Installation terminée")


## Cell 2 — Generation of 120 synthetic calls


In [ ]:
# === CELLULE 2 : Génération des 120 appels synthétiques ===
#
# OBJECTIF : Créer deux fichiers dans le Lakehouse :
#   1. 120 fichiers .txt dans Files/audio_calls/  (simulent des transcriptions)
#   2. Un fichier calls_metadata.csv (horodatage, durée, client, produit)
#
# Dans un projet réel, les .txt seraient remplacés par des .mp3 et la
# transcription serait effectuée par un modèle Whisper ou Azure Speech.
# La structure des données est identique — seule la source change.
#
# Vocabulaire métier SolarVoix :
#   - PSF  = Panneaux Solaires Fotovoltaïques
#   - PAC  = Pompe À Chaleur
#   - BORNE = Borne de recharge VE

import random
import csv
import os
from datetime import datetime, timedelta
from faker import Faker

fake = Faker("fr_FR")
random.seed(42)  # Graine fixe → résultats reproductibles entre sessions

# -------------------------------------------------------
# DÉFINITION DU CORPUS DE TRANSCRIPTIONS
# -------------------------------------------------------
# Chaque catégorie contient des phrases typiques d'un appel client.
# Les catégories couvrent le spectre complet : très satisfait → très mécontent.

TEMPLATES = {
    "positif": [
        "Bonjour, je voulais juste vous remercier pour l'installation de mes panneaux solaires. "
        "Le technicien était très professionnel et tout fonctionne parfaitement. "
        "Je suis vraiment satisfait du service et je vous recommanderai à mes voisins.",

        "Votre équipe a fait un travail remarquable pour ma pompe à chaleur. "
        "L'installation a été rapide, propre, et les économies sur ma facture sont déjà visibles. "
        "Continuez comme ça, c'est exactement ce qu'on attend.",

        "Je viens de recevoir ma première facture EDF depuis l'installation des panneaux. "
        "J'économise déjà 40 pourcent. Votre solution est excellente, merci à toute l'équipe.",
    ],
    "neutre": [
        "Bonjour, je souhaite avoir des informations sur vos offres de maintenance pour panneaux solaires. "
        "Mon contrat expire dans trois mois et je voudrais connaître les options de renouvellement.",

        "Je vous appelle pour signaler un délai de livraison de ma borne de recharge. "
        "On m'avait annoncé 10 jours, il en est au 14ème. Pouvez-vous me donner une date précise ?",

        "Bonjour, j'aimerais avoir une information sur le remboursement de la TVA "
        "pour mon installation de panneaux solaires. Quels documents dois-je fournir ?",
    ],
    "negatif": [
        "Ça fait deux semaines que j'attends un technicien pour réparer ma pompe à chaleur. "
        "C'est inacceptable, j'ai des enfants à la maison et il fait froid. "
        "Si ça continue, je vais déposer une plainte auprès de la DGCCRF.",

        "Mon application de suivi de production solaire ne fonctionne plus depuis 10 jours. "
        "J'ai appelé trois fois, personne ne rappelle. C'est un manque de respect total envers les clients. "
        "Je suis vraiment déçu de votre service après-vente.",

        "Vos techniciens ont endommagé ma toiture lors de la pose des panneaux. "
        "J'attends toujours une réponse de votre responsable depuis deux semaines. "
        "Cette situation est inadmissible et je vais en parler à mon assurance.",
    ],
    "tres_negatif": [
        "Je vais résilier mon contrat immédiatement. Ça fait trois mois que votre borne de recharge "
        "ne fonctionne pas et vous ne faites rien. J'ai perdu confiance en votre entreprise. "
        "Je vais laisser des avis négatifs partout et déposer une plainte formelle.",

        "C'est la dernière fois que je vous appelle. Vous m'avez vendu une pompe à chaleur "
        "qui tombe en panne chaque mois. Je veux un remboursement complet et je vais saisir "
        "le médiateur de l'énergie si vous ne réglez pas ça dans les 48 heures.",

        "Votre entreprise est une arnaque. Mes panneaux solaires n'ont jamais produit "
        "ce que vous m'aviez promis. Je contacte mon avocat cette semaine "
        "et je résilie tous mes contrats avec vous. Vous allez avoir de mes nouvelles.",
    ],
}

PRODUITS = ["Panneaux solaires PSF", "Pompe à chaleur PAC", "Borne recharge VE", "Batterie stockage"]
INTENTIONS = {
    "positif":     ["satisfaction", "recommandation"],
    "neutre":      ["information", "suivi_commande", "question_facturation"],
    "negatif":     ["plainte_technique", "plainte_sav", "reclamation"],
    "tres_negatif":["resiliation", "menace_juridique", "churn_imminent"],
}

# -------------------------------------------------------
# GÉNÉRATION DES 120 APPELS
# -------------------------------------------------------
# Répartition : 30 positifs, 30 neutres, 35 négatifs, 25 très négatifs
# → Représentation réaliste d'un centre SAV sous tension

REPARTITION = (
    [("positif", 30)] +
    [("neutre", 30)] +
    [("negatif", 35)] +
    [("tres_negatif", 25)]
)

appels = []
call_id = 1
date_debut = datetime(2024, 10, 1)

for sentiment, count in REPARTITION:
    for _ in range(count):
        template = random.choice(TEMPLATES[sentiment])
        intention = random.choice(INTENTIONS[sentiment])
        produit = random.choice(PRODUITS)
        duree_sec = random.randint(60, 480)  # 1 à 8 minutes
        date_appel = date_debut + timedelta(
            days=random.randint(0, 89),      # sur 3 mois
            hours=random.randint(8, 18),
            minutes=random.randint(0, 59)
        )
        # Enrichissement du texte avec le produit mentionné
        texte_complet = f"[Appel entrant - {date_appel.strftime('%d/%m/%Y %H:%M')}] " \
                        f"Client : {fake.name()}. Produit concerné : {produit}. " \
                        f"Transcription : {template}"

        appels.append({
            "call_id": f"CALL_{call_id:04d}",
            "client_nom": fake.name(),
            "client_email": fake.email(),
            "ville": fake.city(),
            "produit": produit,
            "date_appel": date_appel.strftime("%Y-%m-%d %H:%M:%S"),
            "duree_secondes": duree_sec,
            "sentiment_reel": sentiment,       # colonne de vérité terrain (pour évaluation)
            "intention_reelle": intention,
            "texte_transcription": texte_complet,
        })
        call_id += 1

random.shuffle(appels)  # Mélanger pour ne pas avoir les sentiments triés
print(f"Nombre d'appels générés : {len(appels)}")
print(f"Répartition sentiments : { {s: sum(1 for a in appels if a['sentiment_reel']==s) for s in TEMPLATES} }")


## Cell 3 — Writing in Lakehouse — Bronze Layer


In [ ]:
# === CELLULE 3 : Écriture dans le Lakehouse (couche Bronze) ===
#
# OBJECTIF : Persister les données générées dans deux formats :
#   1. Fichiers .txt individuels dans Files/audio_calls/
#      → Simulant des transcriptions de fichiers audio réels
#      → En production : ce seraient des .mp3 traités par Whisper ou Azure Speech
#   2. Fichier CSV de métadonnées dans Files/metadata/
#      → Contient toutes les colonnes sauf le texte de transcription
#
# mssparkutils.fs.put() écrit directement dans le système de fichiers OneLake
# du Lakehouse attaché au Notebook.
# overwrite=True → idempotent (peut être relancé sans dupliquer les données)

# -------------------------------------------------------
# ÉCRITURE DES FICHIERS .TXT DE TRANSCRIPTION
# -------------------------------------------------------
for appel in appels:
    chemin = f"Files/audio_calls/{appel['call_id']}.txt"
    mssparkutils.fs.put(chemin, appel["texte_transcription"], overwrite=True)

print(f"✅ {len(appels)} fichiers .txt écrits dans Files/audio_calls/")

# -------------------------------------------------------
# ÉCRITURE DU CSV MÉTADONNÉES
# -------------------------------------------------------
# On utilise Python natif (io + csv) pour construire le CSV en mémoire
# puis mssparkutils.fs.put() pour l'écrire dans le Lakehouse en une seule opération.
# Évite de créer un fichier temporaire local (/tmp/...) qui ne persiste pas.

import io, csv as csv_module

buffer = io.StringIO()
champs = ["call_id","client_nom","client_email","ville","produit",
          "date_appel","duree_secondes","sentiment_reel","intention_reelle"]
writer = csv_module.DictWriter(buffer, fieldnames=champs, extrasaction="ignore")
writer.writeheader()
writer.writerows(appels)

mssparkutils.fs.put("Files/metadata/calls_metadata.csv", buffer.getvalue(), overwrite=True)
print("✅ calls_metadata.csv écrit dans Files/metadata/")
